# Lab 1 · A. Validación de conexión con Azure
**Proyecto Integrador:** Copiloto Cognitivo de Asistencia Bancaria · **Autor:** jorgecr1991

Objetivo: comprobar que el equipo local accede a los recursos asignados usando **Microsoft Entra ID + Azure CLI**,
sin API keys en el código ni en el `.env`.

**Antes de ejecutar:** iniciar sesión en el portal con la cuenta asignada (Cuenta profesional o educativa), configurar Microsoft Authenticator
y ejecutar `az login` en la terminal de VS Code.

In [1]:
# Carga de configuración (solo endpoints y nombres; NUNCA API keys)
import os
from pathlib import Path
from dotenv import load_dotenv

CANDIDATOS_ENV = [
    Path(".env"),  # labs/lab01/.env (opcional, propio del laboratorio)
    Path("../../8.- Material Clases/1.- Nivelacion/2.- Cognitive AI Foundations/Notebooks/.env"),  # .env del curso
]
ENV_USADO = next((p for p in CANDIDATOS_ENV if p.exists()), None)
if ENV_USADO is None:
    raise FileNotFoundError("No se encontró un .env. Copia .env.example a .env y completa los endpoints.")
load_dotenv(ENV_USADO, override=True)
print("Configuración cargada desde:", ENV_USADO.resolve())

# Control de seguridad: el .env no debe contener claves ni secretos con valor
claves = []
for linea in ENV_USADO.read_text(encoding="utf-8").splitlines():
    if "=" in linea and not linea.strip().startswith("#"):
        nombre, valor = linea.split("=", 1)
        if any(t in nombre.upper() for t in ("KEY", "SECRET", "PASSWORD", "TOKEN")) and valor.strip().strip('"'):
            claves.append(nombre.strip())
assert not claves, f"El .env contiene variables tipo clave: {claves}. Elimínalas: la autenticación es con az login."
print("OK: no hay API keys en la configuración; la autenticación será con Microsoft Entra ID (az login).")

Configuración cargada desde: C:\Users\jorge\Documents\Credicorp_Plan de Trabajo_Contenido formativo\8.- Material Clases\1.- Nivelacion\2.- Cognitive AI Foundations\Notebooks\.env
OK: no hay API keys en la configuración; la autenticación será con Microsoft Entra ID (az login).


## Paso 4. Verificar la instalación de Azure CLI (`az --version`)

In [3]:
import shutil, subprocess, json

AZ = shutil.which("az.cmd") or shutil.which("az")
assert AZ, "Azure CLI no está instalada o la terminal no se reinició después de instalarla."
version = json.loads(subprocess.run([AZ, "version", "-o", "json"], capture_output=True, text=True, check=True).stdout)
print("Azure CLI instalada:", version.get("azure-cli"))

Azure CLI instalada: 2.91.0


## Paso 5. Verificar la sesión (`az account show`)
Se enmascaran los identificadores para no publicar IDs en el repositorio.

In [4]:
def enmascarar(valor, visibles=4):
    valor = str(valor or "")
    return valor[:visibles] + "…" + valor[-visibles:] if len(valor) > 2 * visibles else valor

res = subprocess.run([AZ, "account", "show", "-o", "json"], capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("No hay sesión activa. Ejecuta `az login` con la cuenta asignada.\n" + res.stderr)
cuenta = json.loads(res.stdout)
print("Suscripción :", cuenta["name"], "| estado:", cuenta["state"])
print("Usuario     :", enmascarar(cuenta["user"]["name"], 3), "| tipo:", cuenta["user"]["type"])
print("Tenant      :", enmascarar(cuenta["tenantId"]))
print("Sub. ID     :", enmascarar(cuenta["id"]))

Suscripción : Azure subscription 1 | estado: Enabled
Usuario     : jor…com | tipo: user
Tenant      : ec54…9ed6
Sub. ID     : 0e5e…c2c8


## Token de Microsoft Entra ID con `DefaultAzureCredential`
El token **no se imprime**: solo se verifica que se obtiene y cuándo expira.

In [5]:
import time
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default")
print(f"Token obtenido correctamente. Expira en {int((token.expires_on - time.time()) / 60)} minutos.")

Token obtenido correctamente. Expira en 66 minutos.


## Prueba de acceso a los recursos asignados
Si aparece **401/403**, no modificar roles ni políticas: registrar el error y comunicarlo al instructor.

In [6]:
from azure.core.exceptions import HttpResponseError, ClientAuthenticationError
from azure.ai.documentintelligence import DocumentIntelligenceAdministrationClient
from azure.ai.contentsafety import BlocklistClient

resultados = {}

try:
    di_admin = DocumentIntelligenceAdministrationClient(os.environ["DOCUMENT_INTELLIGENCE_ENDPOINT"], credential)
    detalle = di_admin.get_resource_details()
    resultados["Document Intelligence"] = f"OK · modelos personalizados: {detalle.custom_document_models.count}/{detalle.custom_document_models.limit}"
except (HttpResponseError, ClientAuthenticationError) as e:
    resultados["Document Intelligence"] = f"ERROR {getattr(e, 'status_code', '')}: {getattr(e, 'message', e)}"

try:
    cs_blocklists = BlocklistClient(os.environ["CONTENT_SAFETY_ENDPOINT"], credential)
    total = sum(1 for _ in cs_blocklists.list_text_blocklists())
    resultados["Content Safety"] = f"OK · blocklists visibles en el recurso: {total}"
except (HttpResponseError, ClientAuthenticationError) as e:
    resultados["Content Safety"] = f"ERROR {getattr(e, 'status_code', '')}: {getattr(e, 'message', e)}"

for servicio, estado in resultados.items():
    print(f"{servicio:<22} → {estado}")

Document Intelligence  → OK · modelos personalizados: 1/20000
Content Safety         → OK · blocklists visibles en el recurso: 11


## Checkpoint A
- [ ] Acceso al Portal de Azure con la cuenta asignada y Microsoft Authenticator.
- [ ] `az --version` sin errores.
- [ ] `az account show` muestra la suscripción.
- [ ] Token de Entra ID obtenido y ambos servicios responden **OK** sin API keys.

**Registro de incidencias (si aplica):** _anotar aquí cualquier error 401/403 y la hora en que se comunicó al instructor._